In [3]:
import pandas as pd 
import numpy as np
import random
import os
import pickle

from warnings import filterwarnings
from plotnine import ggtitle , ggsave

from scpi_pkg.scdata import scdata
from scpi_pkg.scdataMulti import scdataMulti
from scpi_pkg.scest import scest
from scpi_pkg.scpi import scpi
from scpi_pkg.scplot import scplot
from scpi_pkg.scplotMulti import scplotMulti

In [5]:
# Load Data
df = pd.read_stata("../data/main-sample.dta")

for outcome in ['gdppc23','adj_mad23','adj_transf23']:
    # Data preparation
    id_var = "country"
    outcome_var = outcome
    time_var = "year"
    period_pre = np.arange(1928, 1959)
    period_post = np.arange(1959, 1990) # add one for python range compatibility
    unit_tr="Cuba"
    unit_co=list(set(df[id_var].unique()) - set([unit_tr]))
    cointegrated_data = False
    constant = False

    data_prep = scdata(df=df, id_var=id_var, 
                    outcome_var=outcome_var, 
                    time_var=time_var, 
                    period_pre=period_pre, 
                    period_post=period_post, 
                    unit_tr=unit_tr, 
                    unit_co=unit_co, 
                    constant = constant,
                    cointegrated_data=cointegrated_data)

    # Set options for inference
    u_missp = True
    u_sigma = "HC1"
    u_order = 1
    u_lags = 0
    e_method = "gaussian"
    e_order = 1
    e_lags = 0
    sims = 1000
    cores = 1

    random.seed(666)

    pi_si = scpi(data_prep, sims=sims, w_constr={'name':'simplex',}, u_order=u_order, u_lags=u_lags,
                e_order=e_order, e_lags=e_lags, e_method=e_method, u_missp=u_missp,
                u_sigma=u_sigma, cores=cores)
    print(pi_si)
    if outcome == 'gdppc23':
        gdppc23_results = pi_si
        y_lab = 'GDP per capita'
    elif outcome == 'adj_mad23':
        y_lab = 'Adjusted GDP per capita'
        adj_mad23_results = pi_si
    else:
        y_lab = 'Adjusted GDP per capita minus Soviet aid'
        adj_transf23_results = pi_si

    with open(f'../results/{outcome}_results.pkl', 'wb') as file:
            pickle.dump(pi_si, file)

/var/folders/xd/gksdz50d7r3765bw8t_98y880000gn/T/ipykernel_95165/3189193755.py:2: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.


-----------------------------------------------
Estimating Weights...
Quantifying Uncertainty

 iterations completed (10%)
 iterations completed (20%)
 iterations completed (30%)
 iterations completed (40%)
 iterations completed (50%)
 iterations completed (60%)
 iterations completed (70%)
 iterations completed (80%)
 iterations completed (90%)
-----------------------------------------------------------------------
Call: scpi
Synthetic Control Estimation - Setup

Constraint Type:                      simplex
Constraint Size (Q):                        1
Treated Unit:                               C
Size of the donor pool:                    15
Features                                    1
Pre-treatment period                1928-1958
Pre-treatment periods (used):              31
Adjustment Covariates:                      0

Synthetic Control Estimation - Results

Active donors: 6

Coefficients:
                  Weights
ID   donor               
Cuba Argentina      0.000
     Bolivia 

In [28]:
# Map column prefixes to their result objects
results_map = {
    'gdppc23': gdppc23_results,
    'adj_mad23': adj_mad23_results,
    'adj_transf23': adj_transf23_results
}

# Initialize DF with years (assuming all models share the same timeframe)
years = list(gdppc23_results.period_pre.flatten()) + list(gdppc23_results.period_post.flatten())
df = pd.DataFrame({'year': years})
n_pre = len(gdppc23_results.period_pre)

# Helper lambda to extract flattened list after resetting index
get_data = lambda obj, col: list(obj.reset_index()[col].values.flatten())

for name, res in results_map.items():
    # Observed ('Cuba') and Synthetic ('A')
    df[name] = get_data(res.Y_pre, 'Cuba') + get_data(res.Y_post, 'Cuba')
    df[f'{name}_synth'] = get_data(res.Y_pre_fit, 'A') + get_data(res.Y_post_fit, 'A')
    
    # Loop through CI types to save lines
    ci_configs = [('CI_in_sample', ''), ('CI_all_gaussian', 'gauss')]
    
    for attr, suffix in ci_configs:
        ci_obj = getattr(res, attr).reset_index()
        df[f'{name}_ci{suffix}_low'] = [None] * n_pre + list(ci_obj['Lower'].values.flatten())
        df[f'{name}_ci{suffix}_up'] = [None] * n_pre + list(ci_obj['Upper'].values.flatten())

df.to_stata('../results/scpi-results.dta', write_index=False)